# NGC 4151 Richardson–Lucy spectral deconvolution

This notebook adapts the COSIpy v0.2.1 Crab multi-energy image-deconvolution workflow to the supplied NGC 4151 DC4 files. It reconstructs Galactic sky maps, then performs a point-source Richardson–Lucy spectral unfolding at the known NGC 4151 position and displays the corresponding $E^2dN/dE$ SED.

The supplied source and background histograms are already accumulated in Galactic CDS coordinates. Consequently, the spacecraft orientation is used to build an exposure-integrated Galactic response rather than replaying the original ScAtt-binned workflow literally.

> The cutoff-power-law parameters are obtained with a Poisson forward-folded fit over the complete 100–10,000 keV response. The non-parametric RL points are shown for visualization; weak high-energy bins should not be treated as Gaussian flux measurements.

In [ ]:
from pathlib import Path
import json

import astropy.units as u
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import Table
from numpy.polynomial.legendre import leggauss
from scipy.optimize import minimize
from scipy.special import xlogy

from cosipy.image_deconvolution import (
    build_galactic_response,
    prepare_galactic_histograms,
    run_richardson_lucy_spectral_deconvolution,
    select_orientation_for_pointing_cut,
)
from cosipy.response import FullDetectorResponse, PointSourceResponse

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

## Configuration

The default nside-2 run is intentionally inexpensive and is suitable for validating the workflow. Increase `NSIDE_IMAGE` to 4 or 8 for a better-resolved analysis; response size and construction cost scale with the number of sky pixels. Individual response rows are cached, so interrupted response construction can resume.

In [ ]:
DATA_FILE = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
    "Radio_Quiet_AGN/GammaRay/Paper_Models/"
    "NGC4151_ec_1000_DC4_COSI_cpl_60_fovCut.hdf5"
)
BACKGROUND_FILE = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
    "Radio_Quiet_AGN/DC4_Files/Background/"
    "Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_"
    "NGC4151_60deg_fov_cut.hdf5"
)
DETECTOR_RESPONSE_FILE = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
    "Radio_Quiet_AGN/DC4_Files/"
    "ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered."
    "nonsparse.binnedimaging.imagingresponse.h5"
)
ORIENTATION_FILE = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
    "Radio_Quiet_AGN/DC4_Files/"
    "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
)

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "pyproject.toml").exists():
        REPOSITORY_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from inside the COSIpy repository.")

OUTPUT_DIR = REPOSITORY_ROOT / "outputs/ngc4151_rl_deconvolution"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LARGE_DATA_DIR = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
    "Radio_Quiet_AGN/SED-analysis"
)
LARGE_DATA_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_COORD = SkyCoord(l=155.077, b=75.063, unit="deg", frame="galactic")
FOV_CUT = 60 * u.deg
DATA_CONTAINS_BACKGROUND = False
NSIDE_IMAGE = 2
NSIDE_SCATT_MAP = 4
ITERATIONS = 20
INITIAL_FLUX = 1e-4
SMOOTHING_FWHM = 3 * u.deg
SPECTRAL_ITERATIONS = 1000
INJECTED_NORMALIZATION = 0.15
INJECTED_INDEX = -1.75
INJECTED_CUTOFF_KEV = 1000.0

## Load the data and select the orientation history

The background contains a time axis, while the source histogram is already time integrated. `prepare_galactic_histograms` projects out that background time axis and, by default, forms the observed histogram as source plus background.

In [ ]:
source, background, event = prepare_galactic_histograms(
    DATA_FILE,
    BACKGROUND_FILE,
    data_contains_background=DATA_CONTAINS_BACKGROUND,
)
orientation = select_orientation_for_pointing_cut(
    ORIENTATION_FILE,
    SOURCE_COORD,
    FOV_CUT,
)

source_counts = float(source.to_dense(copy=False).contents.sum())
background_counts = float(background.to_dense(copy=False).contents.sum())
livetime = orientation.cumulative_livetime()

print(f"Source counts:     {source_counts:,.3f}")
print(f"Background counts: {background_counts:,.0f}")
print(f"Selected livetime: {livetime.to_value(u.s):,.0f} s")
print(f"CDS axes:          {event.axes.labels}")

## Build or reuse the Galactic response

This step folds the detector response through the selected orientation history for every Galactic image pixel. Existing response files and cached pixel rows are reused automatically.

In [ ]:
response_path = LARGE_DATA_DIR / (
    f"ngc4151_galactic_response_nside{NSIDE_IMAGE}_scatt{NSIDE_SCATT_MAP}.hdf5"
)
response = build_galactic_response(
    DETECTOR_RESPONSE_FILE,
    orientation,
    response_path,
    nside_image=NSIDE_IMAGE,
    nside_scatt_map=NSIDE_SCATT_MAP,
    earth_occ=True,
)
print(response_path)

## Accelerated multi-energy Richardson–Lucy reconstruction

The model has axes `(Galactic sky pixel, incident energy)`. The update uses MaxStep acceleration, exposure weighting, Gaussian smoothing, and simultaneous optimization of one background normalization.

In [ ]:
algorithm = run_richardson_lucy_spectral_deconvolution(
    event,
    background,
    response,
    iteration_max=ITERATIONS,
    initial_flux=INITIAL_FLUX,
    acceleration_max=5.0,
    response_weighting_index=0.5,
    smoothing_fwhm=SMOOTHING_FWHM,
    background_range=(0.01, 10.0),
    stopping_threshold=0.01,
)
model = algorithm.results[-1]["model"]
model.write(OUTPUT_DIR / "reconstructed_model.hdf5", overwrite=True)
print(f"Completed {len(algorithm.results)} iterations")

In [ ]:
likelihood = np.array([
    np.sum(result["log-likelihood"]) for result in algorithm.results
])
background_norm = np.array([
    result["background_normalization"]["background"]
    for result in algorithm.results
])
iteration_number = np.arange(1, len(algorithm.results) + 1)

figure, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(iteration_number, likelihood, marker="o", ms=3)
axes[0].set(xlabel="Iteration", ylabel="Poisson log-likelihood")
axes[1].plot(iteration_number, background_norm, marker="o", ms=3)
axes[1].set(xlabel="Iteration", ylabel="Background normalization")
plt.show()

print(f"Final background normalization: {background_norm[-1]:.6f}")
print(f"Likelihood monotonic: {np.all(np.diff(likelihood) >= 0)}")

## Reconstructed maps

The red star marks NGC 4151. Dark pixels are zero-exposure or Earth-occulted regions. At nside 2 each HEALPix pixel is large, so this display validates localization but is not a publication-resolution image.

In [ ]:
energy_edges = model.axes["Ei"].edges.to_value(u.keV)
figure, axes = plt.subplots(2, 5, figsize=(20, 8))
for energy_index, axis in enumerate(axes.flat):
    plt.axes(axis)
    hp.mollview(
        model.contents[:, energy_index].value,
        title=(
            f"{energy_edges[energy_index]:g}–"
            f"{energy_edges[energy_index + 1]:g} keV"
        ),
        unit=str(model.unit),
        hold=True,
    )
    hp.projscatter(
        SOURCE_COORD.galactic.l.deg,
        SOURCE_COORD.galactic.b.deg,
        lonlat=True,
        color="red",
        marker="*",
    )
plt.show()

## Point-source spectral deconvolution

Summing every pixel of a weak-source all-sky reconstruction promotes background residuals into source flux. For the SED we instead build the response at the exact NGC 4151 coordinate and apply the Richardson–Lucy/ML-EM update only to the ten incident-energy amplitudes. The supplied simulated background is fixed at normalization one.

In [ ]:
point_response_path = (
    LARGE_DATA_DIR / "ngc4151_exact_point_source_response_scatt4.hdf5"
)
if point_response_path.exists():
    point_response = PointSourceResponse.open(point_response_path)
else:
    with FullDetectorResponse.open(DETECTOR_RESPONSE_FILE, dtype=np.float32) as detector:
        scatt_map = orientation.get_scatt_map(
            nside=NSIDE_SCATT_MAP,
            target_coord=SOURCE_COORD,
            earth_occ=True,
        )
        point_response = detector.get_point_source_response(
            coord=SOURCE_COORD, scatt_map=scatt_map
        )
    point_response.write(point_response_path, overwrite=True)

energy_edges = point_response.axes["Ei"].edges.to_value(u.keV)
energy_centers = point_response.axes["Ei"].centers.to(u.keV)
energy_widths = np.diff(energy_edges)
response_matrix = np.asarray(point_response.contents.value).reshape(10, -1)
event_counts = np.asarray(event.to_dense(copy=False).contents).reshape(-1)
background_model = np.asarray(background.to_dense(copy=False).contents).reshape(-1)
summed_response = response_matrix.sum(axis=1)

band_flux = np.full(10, 1e-4)  # integrated flux in each Ei bin
spectral_log_likelihood = []
for spectral_iteration in range(SPECTRAL_ITERATIONS):
    expectation = background_model + band_flux @ response_matrix
    data_ratio = np.divide(
        event_counts,
        expectation,
        out=np.zeros_like(event_counts),
        where=expectation > 0,
    )
    band_flux *= (response_matrix @ data_ratio) / summed_response
    expectation = background_model + band_flux @ response_matrix
    supported = expectation > 0
    spectral_log_likelihood.append(
        np.sum(
            event_counts[supported] * np.log(expectation[supported])
            - expectation[supported]
        )
    )

flux_values = band_flux / energy_widths
sed_values = energy_centers.to_value(u.keV) ** 2 * flux_values
injected_flux_values = (
    INJECTED_NORMALIZATION
    * energy_centers.to_value(u.keV) ** INJECTED_INDEX
    * np.exp(-energy_centers.to_value(u.keV) / INJECTED_CUTOFF_KEV)
)

spectrum_table = Table(
    {
        "e_min_keV": energy_edges[:-1],
        "e_max_keV": energy_edges[1:],
        "e_ref_keV": energy_centers.to_value(u.keV),
        "dnde_per_cm2_s_keV": flux_values,
        "e2dnde_keV_per_cm2_s": sed_values,
        "injected_dnde_per_cm2_s_keV": injected_flux_values,
    }
)
spectrum_table.write(
    OUTPUT_DIR / "reconstructed_spectrum_and_sed.csv",
    format="ascii.csv",
    overwrite=True,
)
spectrum_table

## Full-range Poisson cutoff-power-law fit

We fit $N(E)=K E^Γ exp(-E/E_c)$ by integrating the model within every incident-energy bin, folding those ten band fluxes through the exact point-source response, adding the fixed background, and evaluating the Poisson likelihood in the complete CDS. Thus every event bin and the full 100–10,000 keV response contribute without pretending that nondetections are precise positive flux points.

In [ ]:
quadrature_nodes, quadrature_weights = leggauss(48)
integration_energy = (
    0.5 * (energy_edges[:-1, None] + energy_edges[1:, None])
    + 0.5 * (energy_edges[1:, None] - energy_edges[:-1, None])
    * quadrature_nodes
)
integration_weights = (
    0.5 * (energy_edges[1:, None] - energy_edges[:-1, None])
    * quadrature_weights
)

def integrated_cutoff_power_law(parameters):
    log_normalization, index, log_cutoff = parameters
    return np.sum(
        integration_weights
        * np.exp(log_normalization)
        * integration_energy**index
        * np.exp(-integration_energy / np.exp(log_cutoff)),
        axis=1,
    )

def poisson_nll(parameters):
    source_expectation = integrated_cutoff_power_law(parameters) @ response_matrix
    expectation = np.maximum(background_model + source_expectation, 1e-300)
    return np.sum(expectation - xlogy(event_counts, expectation))

fit_bounds = (
        (np.log(1e-3), np.log(10.0)),
        (-4.0, 0.0),
        (np.log(100.0), np.log(10000.0)),
)
initial_hypotheses = (
    (0.1, -1.5, 800.0),
    (INJECTED_NORMALIZATION, INJECTED_INDEX, INJECTED_CUTOFF_KEV),
    (0.3, -2.0, 1500.0),
    (0.05, -1.2, 500.0),
    (1.0, -2.2, 3000.0),
)
fit_candidates = [
    minimize(
        poisson_nll,
        x0=(np.log(normalization), index, np.log(cutoff)),
        method="L-BFGS-B",
        bounds=fit_bounds,
        options={"ftol": 1e-14, "gtol": 1e-7, "maxiter": 3000},
    )
    for normalization, index, cutoff in initial_hypotheses
]
fit_result = min(fit_candidates, key=lambda result: result.fun)
fitted_norm = np.exp(fit_result.x[0])
fitted_index = fit_result.x[1]
fitted_cutoff = np.exp(fit_result.x[2])

print(f"Fit range:          {energy_edges[0]:g}–{energy_edges[-1]:g} keV")
print(f"Normalization at 1 keV: {fitted_norm:.4f} cm^-2 s^-1 keV^-1")
print(f"Photon index:       {fitted_index:.4f}")
print(f"Cutoff energy:      {fitted_cutoff:.1f} keV")
print(f"Optimizer success:  {fit_result.success}")

## Photon spectrum and SED

In [ ]:
plot_energy = np.geomspace(energy_edges[0], energy_edges[-1], 400)
fit_curve = fitted_norm * plot_energy**fitted_index * np.exp(
    -plot_energy / fitted_cutoff
)
injected_curve = (
    INJECTED_NORMALIZATION
    * plot_energy**INJECTED_INDEX
    * np.exp(-plot_energy / INJECTED_CUTOFF_KEV)
)
xerr = np.vstack((
    energy_centers.to_value(u.keV) - energy_edges[:-1],
    energy_edges[1:] - energy_centers.to_value(u.keV),
))

figure, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
axes[0].errorbar(
    energy_centers.to_value(u.keV),
    flux_values,
    xerr=xerr,
    fmt="o",
    label="Point-source RL unfolding",
)
axes[0].plot(plot_energy, fit_curve, label="Poisson forward fit")
axes[0].plot(plot_energy, injected_curve, "--", label="Injected CPL")
axes[0].set(
    xscale="log",
    yscale="log",
    xlabel="Energy (keV)",
    ylabel=r"$dN/dE$ (cm$^{-2}$ s$^{-1}$ keV$^{-1}$)",
)
axes[0].legend(fontsize=8)

axes[1].errorbar(
    energy_centers.to_value(u.keV),
    sed_values,
    xerr=xerr,
    fmt="o",
    label=r"Point-source RL $E^2dN/dE$",
)
axes[1].plot(plot_energy, plot_energy**2 * fit_curve, label="Poisson forward fit")
axes[1].plot(
    plot_energy, plot_energy**2 * injected_curve, "--", label="Injected CPL"
)
axes[1].set(
    xscale="log",
    yscale="log",
    xlabel="Energy (keV)",
    ylabel=r"$E^2dN/dE$ (keV cm$^{-2}$ s$^{-1}$)",
)
axes[1].legend(fontsize=8)
figure.savefig(OUTPUT_DIR / "reconstructed_sed_with_fit.png", dpi=180)
plt.show()

In [ ]:
diagnostics = {
    "iterations": len(algorithm.results),
    "livetime_s": livetime.to_value(u.s),
    "source_counts": source_counts,
    "background_counts": background_counts,
    "final_log_likelihood": float(likelihood[-1]),
    "final_background_normalization": float(background_norm[-1]),
    "spectral_rl_iterations": SPECTRAL_ITERATIONS,
    "fit_energy_min_keV": float(energy_edges[0]),
    "fit_energy_max_keV": float(energy_edges[-1]),
    "fit_norm_at_1keV": float(fitted_norm),
    "fit_photon_index": float(fitted_index),
    "fit_cutoff_keV": float(fitted_cutoff),
}
(OUTPUT_DIR / "notebook_diagnostics.json").write_text(
    json.dumps(diagnostics, indent=2)
)
print(f"Results written to {OUTPUT_DIR}")
diagnostics

## Interpretation and next steps

The all-sky maps validate localization, while the point-source RL unfolding provides a non-parametric SED at the known source position. The final four measured-energy source contributions are below 1 sigma over background, so their unfolded points are initialization sensitive. The Poisson forward fit nevertheless uses the complete 100–10,000 keV CDS correctly: those bins constrain the model as nondetections rather than as precise positive flux measurements. Monte Carlo realizations are still required for parameter uncertainties and coverage.